<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day11-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 11 lab: recurrent networks and protein language models {.unnumbered}

This lab has two parts:

- **Part A, sequences in order.** One recurrent step written by hand,
  then per-residue signal-peptide labelling: feed-forward sliding windows
  of growing size against a forward LSTM and a bidirectional LSTM.
- **Part B, protein language-model embeddings.** ESM-2 embeddings of real
  PDB proteins: cosine similarity between pairs, a PCA plot and a
  clustering of eleven proteins, and a check of the clusters against their
  Pfam families.

Every `todo("...")` call marks a piece of code for you to write: replace
the whole `todo(...)` call with your code. Work through the cells **in
order, top to bottom**. Until you fill it in, a cell stops with
`NotImplementedError: TODO in this cell: ...`, which tells you what is
missing. That is expected.

**Runtime:** a few minutes on Colab's free CPU. Part B downloads the
smallest ESM-2 model (8 million parameters, about 30 MB); on Colab, run
`!pip install fair-esm` first. Keep `SEED = 0` so that your numbers match
the lab quiz. Training still varies a little between machines, so the quiz
accepts a range, and you upload your notebook at the end.

In [ ]:
import os, io
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
from sklearn.linear_model import SGDClassifier
from sklearn.metrics import matthews_corrcoef

SEED = 0
torch.manual_seed(SEED)
np.random.seed(SEED)
torch.set_num_threads(max(1, min(4, os.cpu_count() or 1)))

REPO = "https://raw.githubusercontent.com/ElofssonLab/kb8029-book/main/notebooks/data/"
def course_file(name):
    '''Read a course data file from the local repository if present, else from GitHub.'''
    local = os.path.join("data", name)
    return open(local).read() if os.path.exists(local) else requests.get(REPO + name, timeout=60).text

def todo(what):
    """Placeholder for code you write: replace the whole todo(...) call with your own code."""
    raise NotImplementedError(f"TODO in this cell: {what}")

# Part A — Sequences in order

## 1. One recurrent step, by hand

A simple RNN keeps a **hidden state** $h_t$ that is updated at every
position from the current input $x_t$ and the previous hidden state:

$$
h_t = \tanh(W_{ih}\, x_t + b_{ih} + W_{hh}\, h_{t-1} + b_{hh})
$$

The cell below builds a small `nn.RNN`, runs it on a random input of 5
positions, and asks you to reproduce its hidden states with this formula.

In [ ]:
torch.manual_seed(SEED)
rnn = nn.RNN(input_size=4, hidden_size=3, batch_first=True)
x = torch.randn(1, 5, 4)                       # 1 sequence, 5 positions, 4 input features
with torch.no_grad():
    out, _ = rnn(x)                             # PyTorch's hidden state at every position
    W_ih, W_hh, b_ih, b_hh = rnn.weight_ih_l0, rnn.weight_hh_l0, rnn.bias_ih_l0, rnn.bias_hh_l0
    h = torch.zeros(3)                          # h_0
    mine = []
    for t in range(5):
        h = todo('the RNN update for position t')
        mine.append(h)
mine = torch.stack(mine)
print("your hidden states:\n", mine.numpy().round(4))
print("largest difference to nn.RNN:", float((mine - out[0]).abs().max()))

## 2. The data: signal peptides, residue by residue

The Day 11 dataset: 4,910 reviewed UniProt proteins from eukaryotes and
bacteria, 1,936 of them with an experimentally supported signal peptide.
Every one of the first 70 residues gets a label, 1 inside the signal
peptide and 0 outside. The split is homology-aware (Day 8): MMseqs2
clusters at 30% identity, whole clusters per split.

In [ ]:
table = pd.read_csv(io.StringIO(course_file("day11-signal-peptides.tsv")), sep="\t", dtype={"sp_end": "Int64"})
sequences = table.sequence.tolist()
has_sp = table.sp_end.notna().to_numpy().astype(int)
N = 70
labels = []
for s, end in zip(sequences, table.sp_end):
    y = np.zeros(min(len(s), N), dtype=int)
    if pd.notna(end):
        y[:int(end)] = 1
    labels.append(y)

pairs = [l.split("\t") for l in course_file("day11-signal-peptides-mmseqs-30.tsv").strip().split("\n")[1:]]
member_to_rep = {m: r for r, m in pairs}
reps = [member_to_rep.get(a, a) for a in table.accession]
rep_id = {r: i for i, r in enumerate(sorted(set(reps)))}
groups = np.array([rep_id[r] for r in reps])

def homology_split(y, groups, fractions=(0.70, 0.15, 0.15), seed=0):
    '''Assign whole clusters, in random order, to train/val/test (same function as Days 8-11).'''
    rng = np.random.RandomState(seed)
    n_classes = y.max() + 1
    target = np.outer(fractions, np.bincount(y, minlength=n_classes))
    have = np.zeros_like(target)
    assignment = {}
    for g in rng.permutation(np.unique(groups)):
        cls_counts = np.bincount(y[groups == g], minlength=n_classes)
        deficit = ((target - have) * (cls_counts > 0)).sum(axis=1) / target.sum(axis=1)
        best = np.flatnonzero(deficit == deficit.max())
        split = int(rng.choice(best))
        assignment[g] = split
        have[split] += cls_counts
    split_of = np.array([assignment[g] for g in groups])
    return [np.where(split_of == k)[0] for k in range(3)]

train_idx, val_idx, test_idx = homology_split(has_sp, groups)
print(f"{len(sequences)} proteins; train {len(train_idx)} / val {len(val_idx)} / test {len(test_idx)}")
print("training residues (first 70 of each protein):", sum(len(labels[i]) for i in train_idx))

## 3. Feed-forward: a sliding window of growing size

A feed-forward model sees each residue only through a fixed **window**
of neighbours: $w$ residues centred on it, one-hot encoded (20 amino
acids plus one "outside the sequence" symbol). Complete the one-hot
encoding, then train a linear classifier for windows of 1, 5, 11, 21 and
41 residues and score each on the test proteins with the MCC.

In [ ]:
AA = "ACDEFGHIKLMNPQRSTVWY"
AA_IDX = {a: i for i, a in enumerate(AA)}
OUT = 20                                                  # symbol for positions outside the sequence

def window_features(idx, w):
    '''One row per residue: the one-hot encoded window of w residues centred on it.'''
    r = w // 2
    rows, y = [], []
    for i in idx:
        s = sequences[i][:N]
        L = len(s)
        for p in range(L):
            rows.append([AA_IDX.get(s[j], OUT) if 0 <= j < L else OUT for j in range(p - r, p + r + 1)])
            y.append(labels[i][p])
    rows = np.array(rows)
    X = np.zeros((len(rows), w * 21), dtype=np.float32)
    X = todo("set the one-hot entries: row n, window position k, symbol c -> column k*21 + c equals 1")
    return X, np.array(y)

X_check, _ = window_features(train_idx[:2], 3)           # quick check on two proteins
print("feature matrix for two proteins, window 3:", X_check.shape, " ones per row:", X_check.sum(1)[:5])

In [ ]:
window_mcc = {}
for w in [1, 5, 11, 21, 41]:
    X_tr, y_tr = window_features(train_idx, w)
    X_te, y_te = window_features(test_idx, w)
    clf = SGDClassifier(loss="log_loss", alpha=1e-4, class_weight="balanced", max_iter=50, random_state=SEED)
    clf.fit(X_tr, y_tr)
    window_mcc[w] = todo("MCC of the classifier's predictions on the test residues")
    print(f"window {w:2d} residues: test MCC {window_mcc[w]:.3f}")

## 4. Recurrent: a forward LSTM and a bidirectional LSTM

The same task, now with a network that reads the whole 70-residue
N-terminus: an embedding, an LSTM, and a linear layer giving one logit
per residue. Complete the output layer: a bidirectional LSTM concatenates
a forward and a backward hidden state at every position.

In [ ]:
class Tagger(nn.Module):
    def __init__(self, bidirectional, emb=24, hidden=48):
        super().__init__()
        self.emb = nn.Embedding(21, emb)
        self.lstm = nn.LSTM(emb, hidden, batch_first=True, bidirectional=bidirectional)
        out_features = todo('size of the LSTM output at each position')
        self.out = nn.Linear(out_features, 1)

    def forward(self, x):
        return self.out(self.lstm(self.emb(x))[0]).squeeze(-1)

print(Tagger(bidirectional=True))                        # quick check that the class builds

In [ ]:
def encode(idx):
    X = np.full((len(idx), N), OUT); Y = np.zeros((len(idx), N), np.float32); M = np.zeros((len(idx), N), np.float32)
    for k, i in enumerate(idx):
        for p, ch in enumerate(sequences[i][:N]):
            X[k, p], Y[k, p], M[k, p] = AA_IDX.get(ch, OUT), labels[i][p], 1.0
    return torch.tensor(X), torch.tensor(Y), torch.tensor(M)

X_tr_seq, Y_tr_seq, M_tr_seq = encode(train_idx)
X_te_seq, Y_te_seq, M_te_seq = encode(test_idx)

rnn_mcc = {}
for name, bi in [("forward LSTM", False), ("bidirectional LSTM", True)]:
    torch.manual_seed(SEED)
    model = Tagger(bidirectional=bi)
    opt = torch.optim.Adam(model.parameters(), lr=1e-3)
    for epoch in range(8):
        order = torch.randperm(len(train_idx), generator=torch.Generator().manual_seed(SEED + epoch))
        for b in order.split(64):
            opt.zero_grad()
            loss = nn.functional.binary_cross_entropy_with_logits(model(X_tr_seq[b]), Y_tr_seq[b], reduction="none")
            ((loss * M_tr_seq[b]).sum() / M_tr_seq[b].sum()).backward()
            opt.step()
    with torch.no_grad():
        pred = (torch.sigmoid(model(X_te_seq)) > 0.5).numpy()
    keep = M_te_seq.numpy() > 0
    rnn_mcc[name] = matthews_corrcoef(Y_te_seq.numpy()[keep], pred[keep])
    print(f"{name:20s}: test MCC {rnn_mcc[name]:.3f}")

# Part B — Protein language-model embeddings

## 5. ESM-2 embeddings and cosine similarity

ESM-2 is a Transformer trained by masked-residue prediction on millions
of UniRef sequences. For every residue it outputs a vector (320 numbers
for the smallest model). Averaging those vectors over the sequence gives
one **embedding** per protein. Complete the averaging, then compute the
cosine similarity between three proteins from the PDB: sperm-whale
myoglobin (104M), pig myoglobin (1MWC), and a bacterial manganese
transporter (10LE).

In [ ]:
try:
    import esm
except ImportError:                              # e.g. on Colab: install the ESM package once
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "fair-esm"], check=True)
    import esm
esm_model, alphabet = esm.pretrained.esm2_t6_8M_UR50D()
esm_model.eval()
batch_converter = alphabet.get_batch_converter()

def embed(seq):
    '''Mean ESM-2 embedding (last layer) over the residues of seq.'''
    _, _, tokens = batch_converter([("p", seq)])
    with torch.no_grad():
        rep = esm_model(tokens, repr_layers=[esm_model.num_layers])["representations"][esm_model.num_layers][0]
    return todo('average the residue vectors (skip the start token at index 0)')

print("embedding length:", embed("MKTAYIAKQR").shape)

In [ ]:
seq_a = "VLSEGEWQLVLHVWAKVEADVAGHGQDILIRLFKSHPETLEKFDRFKHLKTEAEMKASEDLKKHGVTVLTALGAILKKKGHHEAELKPLAQSHATKHKIPIKYLEFISEAIIHVLHSRHPGDFGADAQGAMNKALELFRKDIAAKYKELGYQG"   # 104M
seq_b = "GLSDGEWQLVLNVWGKVEADVAGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGNTVLTALGGILKKKGHHEAELTPLAQSHATKHKIPVKYLEFISEAIIQVLQSKHPGDFGADAQGAMSKALELFRNDMAAKYKELGFQG"   # 1MWC
seq_c = "MHHHHHHHHAHMKRYLGGLDVFRYIGPGLLVTVGFIDPGNWASNFAAGSEFGYSLLWVVTLSTIMLIILQHNVAHLGIVTGLCLSEAATQYTPKWVSRPILGTAVLASISTSLAEILGGAIALEMLLDIPIVWGAVLTTVFVSIMLFTNSYKKIERSIIAFVSVIGLSFIYELFLVDIDWPMAVEGWVTPAIPKGSMLIIMSVLGAVVMPHNLFLHSEVIQSHEYNKQDTASIKKVLKYELFDTLFSMIIGWAINSAMILLAAATFFKSGIQVEELQQAKSLLEPLLGSNAAIVFALALLMAGISSTITSGMAAGSIFAGIFGESYHIKDSHSQVGVILSLGIALLLIFFIGDPFKGLIISQMVLSIQLPFTVFLQVGLTSSRKVMGDYVNSKWSTFVLYTIAVIVTVLNIMLLFS"   # 10LE

def cosine(u, v):
    return todo('cosine similarity of vectors u and v')

e_a, e_b, e_c = embed(seq_a), embed(seq_b), embed(seq_c)
print(f"cosine(A, B) = {cosine(e_a, e_b):.3f}   (104M vs 1MWC)")
print(f"cosine(A, C) = {cosine(e_a, e_c):.3f}   (104M vs 10LE)")
print(f"cosine(B, C) = {cosine(e_b, e_c):.3f}   (1MWC vs 10LE)")

## 6. Eleven proteins: PCA, clusters and Pfam

The sequences of eleven PDB proteins are fetched live from RCSB. Embed
them, project the embeddings to two dimensions with PCA, and cluster them
(average linkage on cosine distance). Then compare the clusters with each
protein's Pfam family, fetched live from InterPro.

In [ ]:
PROTEINS = ["104M_1", "1MWC_1", "4HHB_1", "4HHB_2", "2PTN_1", "3EST_1", "1HNE_1", "1A0J_1",
            "1HEL_1", "1REX_1", "1HFY_1"]
info = {}
for pid in PROTEINS:
    entry = pid.split("_")[0]
    for rec in requests.get(f"https://www.rcsb.org/fasta/entry/{entry}", timeout=30).text.split(">")[1:]:
        header, seq = rec.split("\n", 1)
        if header.split("|")[0] == pid:
            info[pid] = (header.split("|")[2], header.split("|")[3], seq.replace("\n", ""))
for pid, (name, organism, seq) in info.items():
    print(f"{pid:7s} {len(seq):4d} aa  {name} ({organism})")

X = np.stack([embed(info[p][2]) for p in PROTEINS])

In [ ]:
from sklearn.decomposition import PCA
xy = PCA(n_components=2, random_state=0).fit_transform(X)
plt.figure(figsize=(6, 5))
plt.scatter(xy[:, 0], xy[:, 1], s=40)
for (px, py), pid in zip(xy, PROTEINS):
    plt.annotate(pid, (px, py), fontsize=8, xytext=(3, 3), textcoords="offset points")
plt.xlabel("PC 1"); plt.ylabel("PC 2"); plt.title("ESM-2 embeddings of eleven PDB proteins (PCA)")
plt.show()

In [ ]:
from scipy.cluster.hierarchy import linkage, fcluster
Z = linkage(X, method="average", metric="cosine")
clusters = todo('cut the tree into 3 clusters (scipy fcluster, criterion "maxclust")')

def pfam(pdb_id):
    j = requests.get(f"https://www.ebi.ac.uk/interpro/api/entry/pfam/structure/pdb/{pdb_id.lower()}", timeout=30).json()
    return ", ".join(f"{r['metadata']['accession']} {r['metadata']['name']}" for r in j["results"])

for c in sorted(set(clusters)):
    print(f"cluster {c}:")
    for pid, k in zip(PROTEINS, clusters):
        if k == c:
            print(f"   {pid:7s} {info[pid][0]:32s} Pfam: {pfam(pid.split('_')[0])}")

**Last step:** save your notebook with all outputs (File → Download →
.ipynb) and upload it with the lab quiz on Canvas. If one of your numbers
falls outside the quiz's accepted range, the notebook lets us see whether
that is ordinary run-to-run variation in training.